# Phase 6 — Feature Engineering and Preprocessing
## Parkinson's Disease Prediction Using Typing Characteristics

**Dataset:** Tappy Extended Keystroke Dataset, Mendeley Data Version 3 (DOI: `10.17632/z39mhdsynx.3`)  
**Working Branch:** `praneeth`  
**Strict Scope Boundary:** Phase 6 implements deterministic feature extraction from validated accepted records (Case 2 resolution) and reusable, unfitted preprocessing components. **No model fitting, classifier training, or performance evaluation is conducted here.**

In [1]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Configure paths relative to repository root
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.feature_engineering.config import FeatureEngineeringConfig
from src.feature_engineering.feature_schema import (
    get_feature_definitions,
    get_predictor_columns,
    get_quality_columns,
    validate_predictor_allowlist,
)
from src.feature_engineering.feature_extraction import (
    extract_participant_features,
    extract_participant_quality,
    aggregate_cohort_features,
)
from src.feature_engineering.quality import (
    validate_feature_table,
    validate_quality_table,
    generate_feature_quality_summary,
)
from src.feature_engineering.preprocessing import build_preprocessing_pipeline

print(f'Feature engineering environment initialized. Project root: {repo_root}')

Feature engineering environment initialized. Project root: C:\Users\prane\.vscode\extensions\ms-vscode.cmake-tools-1.21.36\ml project\PARKINSONS-DISEASE-PREDICTOR


## 1. Load Configuration & Verified Phase 4 Cohort
We load the configuration and verify the 597 eligible participants from the Phase 4 train/test split manifests.

In [2]:
config = FeatureEngineeringConfig.load(repo_root=repo_root)
train_df = pd.read_csv(config.train_ids_path)
test_df = pd.read_csv(config.test_ids_path)
eligible_ids = sorted(train_df['participant_id'].tolist() + test_df['participant_id'].tolist())
labels_df = pd.read_csv(config.participant_labels_path)

print(f'Training partition count: {len(train_df)}')
print(f'Test partition count: {len(test_df)}')
print(f'Total eligible cohort count: {len(eligible_ids)}')
assert len(eligible_ids) == config.expected_eligible_count

Training partition count: 477
Test partition count: 120
Total eligible cohort count: 597


## 2. Source Record Inspection & Schema Verification
We inspect the accepted monthly Parquet files produced by Phase 2 in `data/interim/accepted/`.

In [3]:
sample_file = list(config.interim_accepted_dir.glob('*.parquet'))[0]
df_sample = pd.read_parquet(sample_file)
print(f'Sample monthly file: {sample_file.name}')
print(f'Columns: {df_sample.columns.tolist()}')
print(f'Record count: {len(df_sample)}')
print(df_sample.head(3))

Sample monthly file: 04AUPVMQXY_1708.parquet
Columns: ['source_file', 'line_no', 'user_key', 'event_time', 'hand', 'hold_ms', 'direction', 'latency_ms', 'flight_ms', 'warning_codes']
Record count: 63
           source_file  line_no    user_key              event_time hand  \
0  04AUPVMQXY_1708.txt        1  04AUPVMQXY 2017-08-23 08:12:27.529    L   
1  04AUPVMQXY_1708.txt        2  04AUPVMQXY 2017-08-23 08:12:27.738    L   
2  04AUPVMQXY_1708.txt        3  04AUPVMQXY 2017-08-23 08:12:28.002    R   

   hold_ms direction  latency_ms  flight_ms warning_codes  
0    103.5        LL       375.0      263.7                
1    103.5        LL       209.0      105.5                
2     80.1        LR       287.1      183.6                


## 3. Feature Catalog & Allowlist Governance
We verify the complete catalog of 153 typing predictors and ensure that no identifiers, labels, or clinical covariates can enter the predictor matrix.

In [4]:
definitions = get_feature_definitions()
predictor_cols = get_predictor_columns()
print(f'Total cataloged features: {len(definitions)}')
print(f'Total predictor columns: {len(predictor_cols)}')

is_valid, violations = validate_predictor_allowlist(predictor_cols)
assert is_valid, f'Prohibited columns found: {violations}'
print('Predictor allowlist verified: zero prohibited covariates or identifiers.')

Total cataloged features: 153
Total predictor columns: 153
Predictor allowlist verified: zero prohibited covariates or identifiers.


## 4. Cohort Feature Extraction Verification
We load the unscaled participant-level feature table (`data/processed/features/participant_features_raw.parquet`) and quality metadata (`participant_feature_quality.parquet`).

In [5]:
raw_features_path = config.features_output_dir / config.raw_features_file
quality_features_path = config.features_output_dir / config.quality_features_file

df_features = pd.read_parquet(raw_features_path)
df_quality = pd.read_parquet(quality_features_path)

print(f'Feature table shape: {df_features.shape}')
print(f'Quality table shape: {df_quality.shape}')
print(f'Class breakdown: PD={int((df_features["parkinsons_label"]==1).sum())}, Control={int((df_features["parkinsons_label"]==0).sum())}')
assert len(df_features) == 597
assert len(df_quality) == 597

Feature table shape: (597, 155)
Quality table shape: (597, 11)
Class breakdown: PD=432, Control=165


## 5. Feature Table Quality Audit
We execute deterministic validation checks on the unscaled feature table:
1. 597 unique participants matching the eligible cohort.
2. Exact target label agreement with authoritative metadata.
3. Zero date corruption signatures (>100,000 integer dates).
4. Zero infinite values.
5. Rollover flight times preserved without distortion.

In [6]:
feat_val = validate_feature_table(df_features, expected_count=597)
qual_val = validate_quality_table(df_quality, expected_count=597)

print('--- Feature Table Validation ---')
print(f'Status: {"PASSED" if feat_val["passed"] else "FAILED"}')
print(f'Issues: {feat_val["issues"]}')
print(f'Date-corrupted columns: {len(feat_val["date_corrupted_columns"])}')
print(f'Infinite columns: {len(feat_val["infinite_columns"])}')

print('\n--- Quality Table Validation ---')
print(f'Status: {"PASSED" if qual_val["passed"] else "FAILED"}')
print(f'Total events aggregated: {qual_val["total_records_sum"]:,}')
print(f'Total warning-tagged records retained: {qual_val["total_warned_records_sum"]:,}')

--- Feature Table Validation ---
Status: PASSED
Issues: []
Date-corrupted columns: 0
Infinite columns: 0

--- Quality Table Validation ---
Status: PASSED
Total events aggregated: 39,503,643
Total warning-tagged records retained: 298,271


## 6. Preprocessing Components & Synthetic Validation
We verify the reusable, **unfitted** scikit-learn preprocessing pipelines (`SimpleImputer`, `StandardScaler`, `RobustScaler`, `PowerTransformer`).
**Crucial Safeguard:** The pipeline remains unfitted in Phase 6. We test it on synthetic feature data to verify that transformations operate correctly.

In [7]:
# Build an unfitted standard scaling pipeline
sample_cols = ['hold_mean', 'hold_std', 'latency_mean', 'flight_mean']
pipe_standard = build_preprocessing_pipeline(
    scaler='standard',
    imputer_strategy='median',
    predictor_cols=sample_cols,
)

# Build an unfitted tree-model passthrough pipeline
pipe_tree = build_preprocessing_pipeline(
    scaler='none',
    imputer_strategy='median',
    predictor_cols=sample_cols,
)

# Test on synthetic data
np.random.seed(42)
synth_data = pd.DataFrame({
    'hold_mean': [100.0, 110.0, np.nan, 130.0, 140.0],
    'hold_std': [30.0, 35.0, 40.0, 45.0, np.nan],
    'latency_mean': [300.0, 350.0, 400.0, 450.0, 500.0],
    'flight_mean': [100.0, 120.0, 140.0, 160.0, 180.0],
})

synth_transformed = pipe_standard.fit_transform(synth_data)
print('Synthetic data transformed cleanly with standard scaler:')
print(synth_transformed)
synth_arr = synth_transformed.to_numpy() if hasattr(synth_transformed, 'to_numpy') else np.asarray(synth_transformed)
assert not np.isnan(synth_arr).any()

Synthetic data transformed cleanly with standard scaler:
   hold_mean  hold_std  latency_mean  flight_mean
0  -1.414214      -1.5     -1.414214    -1.414214
1  -0.707107      -0.5     -0.707107    -0.707107
2   0.000000       0.5      0.000000     0.000000
3   0.707107       1.5      0.707107     0.707107
4   1.414214       0.0      1.414214     1.414214


## 7. Phase 6 Deliverables & Manifest Verification
We inspect the authoritative `feature_manifest.json` generated by Phase 6.

In [8]:
manifest_path = config.features_output_dir / config.manifest_file
with open(manifest_path, 'r', encoding='utf-8') as f:
    manifest = json.load(f)

print('--- Feature Manifest Summary ---')
print(f'Dataset: {manifest["dataset_name"]}')
print(f'Cohort size: {manifest["cohort"]["eligible_participant_count"]}')
print(f'Predictor count: {manifest["feature_counts"]["predictor_feature_count"]}')
print(f'Raw features Parquet: {manifest["artifacts"]["raw_features_parquet"]["path"]}')
print(f'SHA-256: {manifest["artifacts"]["raw_features_parquet"]["sha256"]}')
print(f'Integrity verifications: {manifest["integrity_verifications"]}')

--- Feature Manifest Summary ---
Dataset: Tappy Extended Keystroke Dataset, Mendeley Data Version 3
Cohort size: 597
Predictor count: 153
Raw features Parquet: data\processed\features\participant_features_raw.parquet
SHA-256: 68a22acba34901549f3dd55e4e52dcd8a683b41c75666c41187e26736259e0ef
Integrity verifications: {'all_597_eligible_covered': True, 'zero_date_corruption_gt_100k': True, 'negative_flight_times_preserved': True, 'zero_infinite_values': True, 'prohibited_covariates_excluded': True, 'test_set_isolation_respected': True}


## 8. Summary & Strict Boundary Enforcement
1. **Re-Extraction Complete:** 597 eligible participants aggregated from clean accepted records.
2. **Data Cleanliness:** 0 date corruption values, 0 infinite values, 100% label concordance.
3. **Preprocessing Modularity:** Reusable unfitted pipelines ready for Phase 7 CV integration.
4. **Strict Hard Stop:** No models fitted, no classifiers instantiated. Phase 6 is complete.